# 7-Day Kerala Coastal Weather, Cyclone Risk & District Alert Zone Prediction System

### Data Preprocessing

In [1]:
import pandas as pd
import numpy as np


In [2]:
#reading the dataset
df=pd.read_csv('open-meteo-8.54N76.92E12m.csv',skiprows=3)#to skip 3 rows
df.head()

,time,surface_pressure (hPa),temperature_2m (°C),rain (mm),wind_speed_10m (km/h),wind_gusts_10m (km/h)
0,2015-01-01T00:00,1008.4,24.4,0.20,6.6,10.4
1,2015-01-01T01:00,1007.8,24.3,0.00,6.2,12.2
2,2015-01-01T02:00,1007.2,24.1,0.00,6.7,13.3
3,2015-01-01T03:00,1007.2,23.7,0.20,6.1,15.5
4,2015-01-01T04:00,1007.3,23.8,0.20,6.1,15.5


In [3]:
#checking for null values
df.isna().sum()

time                         0
surface_pressure (hPa)       0
temperature_2m (°C)          0
rain (mm)                    0
wind_speed_10m (km/h)     4019
wind_gusts_10m (km/h)     4019
dtype: int64

In [4]:
df.shape

(100451, 6)

In [5]:
#to get the statsical analysis
df.describe()

,wind_speed_10m (km/h),wind_gusts_10m (km/h)
count,96432.000000,96432.000000
mean,9.733015,22.838353
std,5.638797,12.402271
min,0.000000,0.700000
25%,5.200000,12.600000
50%,8.700000,20.900000
75%,13.600000,31.700000
max,37.600000,77.400000


In [6]:
#filling the missing columns with median for two columns
df['wind_speed_10m (km/h)']=df['wind_speed_10m (km/h)'].fillna(df['wind_speed_10m (km/h)'].median())
df['wind_gusts_10m (km/h)']=df['wind_gusts_10m (km/h)'].fillna(df['wind_gusts_10m (km/h)'].median())

In [7]:
df.isna().sum()

time                      0
surface_pressure (hPa)    0
temperature_2m (°C)       0
rain (mm)                 0
wind_speed_10m (km/h)     0
wind_gusts_10m (km/h)     0
dtype: int64

In [12]:
df.dtypes

time                          str
surface_pressure (hPa)        str
temperature_2m (°C)           str
rain (mm)                     str
wind_speed_10m (km/h)     float64
wind_gusts_10m (km/h)     float64
dtype: object

In [13]:
#here we have found out that 3 columns are string and we need to convert into numeric
cols_to_convert=['surface_pressure (hPa)','temperature_2m (°C)','rain (mm)']
for col in cols_to_convert:
    df[col]=pd.to_numeric(df[col],errors='coerce')

## Here we have two targets 


### 1.Regression Target: Flood Runoff (80% of rainfall volume)

In [9]:
#Flood_Runoff_mm (Regression Target)
#What it represents: A continuous numerical value showing the estimated amount of water (in millimeters) that turns into surface runoff rather than soaking into the soil.
#How it is calculated: It takes 80% of the hourly rainfall volume (rain (mm) * 0.80), reflecting standard hydrological practices for tropical, coastal regions like Kerala where high soil saturation leads to significant runoff during heavy precipitation.


In [14]:
df['Flood_Runoff_mm'] = df['rain (mm)'] * 0.80

In [15]:
df.dtypes

time                          str
surface_pressure (hPa)    float64
temperature_2m (°C)       float64
rain (mm)                 float64
wind_speed_10m (km/h)     float64
wind_gusts_10m (km/h)     float64
Flood_Runoff_mm           float64
dtype: object

In [16]:
df.isna().sum() #checking for null values

time                      0
surface_pressure (hPa)    1
temperature_2m (°C)       1
rain (mm)                 1
wind_speed_10m (km/h)     0
wind_gusts_10m (km/h)     0
Flood_Runoff_mm           1
dtype: int64

In [ ]:
#removing the null values of columns surface_pressure (hPa),temperature_2m (°C), rain (mm), Flood_Runoff_mm  

In [17]:
df['surface_pressure (hPa)']=df['surface_pressure (hPa)'].fillna(df['surface_pressure (hPa)'].mean())
df['temperature_2m (°C)']=df['temperature_2m (°C)'].fillna(df['temperature_2m (°C)'].mean())
df['rain (mm)']=df['rain (mm)'].fillna(0)
df['Flood_Runoff_mm']=df['Flood_Runoff_mm'].fillna(0)


In [18]:
df['Flood_Runoff_mm'] = df['rain (mm)'] * 0.80

In [19]:
df

,time,surface_pressure (hPa),temperature_2m (°C),rain (mm),wind_speed_10m (km/h),wind_gusts_10m (km/h),Flood_Runoff_mm
0,2015-01-01T00:00,1008.4,24.4,0.2,6.6,10.4,0.16
1,2015-01-01T01:00,1007.8,24.3,0.0,6.2,12.2,0.00
2,2015-01-01T02:00,1007.2,24.1,0.0,6.7,13.3,0.00
3,2015-01-01T03:00,1007.2,23.7,0.2,6.1,15.5,0.16
4,2015-01-01T04:00,1007.3,23.8,0.2,6.1,15.5,0.16
...,...,...,...,...,...,...,...
100446,2025-12-27,0.0,1009.3,1011.2,8.7,20.9,808.96
100447,2025-12-28,1.2,1010.1,1011.7,8.7,20.9,809.36
100448,2025-12-29,0.0,1009.6,1011.5,8.7,20.9,809.20
100449,2025-12-30,0.3,1009.9,1012.3,8.7,20.9,809.84


### 2. Classification Target: Alert Zone (IMD Weather Thresholds)

#### What it represents: A categorical integer label (0, 1, 2, or 3) that translates raw weather measurements into an official risk level based on India Meteorological Department (IMD) thresholds.
##### Categorized as:
   ##### 0 (Green Zone — Safe): Low rainfall (< 20 mm) and calm winds (< 40 km/h). Normal daily conditions.
   ##### 1 (Yellow Zone — Watch/Advisory): Moderate rainfall (between 20 and 64.5 mm), signaling a weather watch.
   ##### 2 (Orange Zone — Heavy Alert): Heavy rainfall (between 64.5 and 115.5 mm), indicating potential localized waterlogging or minor flooding.
   ##### 3 (Red Zone — Severe Warning / Cyclone): Extreme rainfall ($\ge 115.5$ mm) or high wind speeds ($\ge 62$ km/h), representing severe hazard risks like severe storms, heavy floods, or cyclonic conditions.

In [20]:
conditions = [
    (df['rain (mm)'] < 20) & (df['wind_speed_10m (km/h)'] < 40),
    (df['rain (mm)'] >= 20) & (df['rain (mm)'] < 64.5),
    (df['rain (mm)'] >= 64.5) & (df['rain (mm)'] < 115.5),
    (df['rain (mm)'] >= 115.5) | (df['wind_speed_10m (km/h)'] >= 62)
]
choices = [0, 1, 2, 3]

In [21]:
df['Alert_Zone'] = np.select(conditions, choices, default=1)


In [22]:
df

,time,surface_pressure (hPa),temperature_2m (°C),rain (mm),wind_speed_10m (km/h),wind_gusts_10m (km/h),Flood_Runoff_mm,Alert_Zone
0,2015-01-01T00:00,1008.4,24.4,0.2,6.6,10.4,0.16,0
1,2015-01-01T01:00,1007.8,24.3,0.0,6.2,12.2,0.00,0
2,2015-01-01T02:00,1007.2,24.1,0.0,6.7,13.3,0.00,0
3,2015-01-01T03:00,1007.2,23.7,0.2,6.1,15.5,0.16,0
4,2015-01-01T04:00,1007.3,23.8,0.2,6.1,15.5,0.16,0
...,...,...,...,...,...,...,...,...
100446,2025-12-27,0.0,1009.3,1011.2,8.7,20.9,808.96,3
100447,2025-12-28,1.2,1010.1,1011.7,8.7,20.9,809.36,3
100448,2025-12-29,0.0,1009.6,1011.5,8.7,20.9,809.20,3
100449,2025-12-30,0.3,1009.9,1012.3,8.7,20.9,809.84,3


In [23]:
print("Alert Zone Distribution:\n", df['Alert_Zone'].value_counts())

Alert Zone Distribution:
 Alert_Zone
0    96427
3     4018
1        6
Name: count, dtype: int64


## No need for outlier analysis

#### 1. Weather "Outliers" Are Actually Storms!
##### In normal datasets, weird numbers (outliers) are usually typos or mistakes that you want to delete.
##### But in weather data, a massive spike in rain or wind isn't a mistake it's a heavy storm or cyclone. If you delete those numbers, your model will never learn what a Red or Orange Alert actually looks like!

#### 2. Random Forest Doesn’t Get Confused by Big Numbers
##### Some machine learning models get thrown off by extreme numbers because they rely on rigid mathematical formulas.
##### Random Forest works differently. It makes decisions using simple rules (like Is rainfall greater than 120 mm?). It simply handles big storms by putting them in their own separate decision path without messing up the normal data.

#### 3. It Keeps People Safe (High Recall)
##### Since your project is a safety warning system for coastal areas and fisherfolk, your main goal is to never miss a dangerous day. Keeping all the extreme weather data ensures the model stays sharp when a real hazard hits.


## Splitting

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100451 entries, 0 to 100450
Data columns (total 8 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   time                    100451 non-null  str    
 1   surface_pressure (hPa)  100451 non-null  float64
 2   temperature_2m (°C)     100451 non-null  float64
 3   rain (mm)               100451 non-null  float64
 4   wind_speed_10m (km/h)   100451 non-null  float64
 5   wind_gusts_10m (km/h)   100451 non-null  float64
 6   Flood_Runoff_mm         100451 non-null  float64
 7   Alert_Zone              100451 non-null  int64  
dtypes: float64(6), int64(1), str(1)
memory usage: 6.1 MB


In [25]:
from sklearn.model_selection import train_test_split
#defining features and target
features=['surface_pressure (hPa)','temperature_2m (°C)','rain (mm)','wind_speed_10m (km/h)','wind_gusts_10m (km/h)','Flood_Runoff_mm']
X = df[features]
y = df['Alert_Zone']
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=42,stratify=y)


## Model Training

##### Here is why tree-based algorithms like Random Forest are uniquely suited for meteorological and disaster prediction projects
#### 1. Handles Non-Linear Weather Dynamics
##### Weather systems do not follow simple straight-line rules. For example, a slight drop in surface pressure combined with a moderate rise in wind speed might be completely safe on its own, but when paired with heavy rainfall, it triggers a dangerous flash flood
##### Random Forest builds multiple decision trees that excel at capturing these complex, non-linear interactions between temperature, pressure, wind, and rain.

#### 2. No Feature Scaling Required
##### Columns are different scales 
##### Algorithms like Support Vector Machines (SVM) or Logistic Regression require you to normalize or scale all these numbers to the same range so one doesn't overpower the other. Random Forest is scale-invariant—it splits data based on thresholds, so the raw units don't matter at all.

#### 3.Naturally Immune to Extreme Storm Data (Outliers)
##### As discussed earlier, heavy monsoon cloudbursts and cyclones create massive spikes in your data. While distance-based models (like KNN or SVM) get distorted by these extreme values, Random Forest handles them effortlessly by sorting them into dedicated split branches without affecting normal data points.

#### 4. Built-in Feature Importance
##### For a public safety or hazard alert system, stakeholders or disaster management authorities often want to know why an alert was triggered. Random Forest allows you to easily extract Feature Importances to show whether rainfall, wind gusts, or pressure drops had the biggest impact on predicting a Red

#### 5. Prevents Overfitting
##### A single Decision Tree can easily "memorize" the training data (overfitting) and perform poorly on new data. Random Forest solves this by combining hundreds of trees (an ensemble) trained on random subsets of the data, averaging their votes for a much more stable, accurate prediction.


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report

In [27]:
#Initialize and train the Random Forest Classifier
print("Training Random Forest model")
model=RandomForestClassifier(n_estimators=100, random_state=42) #Creating an object
model.fit(X_train,y_train)


Training Random Forest model


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [28]:
# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {accuracy * 100:.2f}%\n")


Overall Model Accuracy: 100.00%



In [29]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19286
           1       1.00      1.00      1.00         1
           3       1.00      1.00      1.00       804

    accuracy                           1.00     20091
   macro avg       1.00      1.00      1.00     20091
weighted avg       1.00      1.00      1.00     20091



In [45]:
cm = confusion_matrix(y_test, y_pred)

In [48]:
cm

array([[19286,     0,     0],
       [    0,     1,     0],
       [    0,     0,   804]])

In [30]:
import joblib

In [31]:
model_filename = 'kerala_weather_model.pkl' #creating a file name
joblib.dump(model, model_filename)

print(f"Model successfully saved as '{model_filename}'!")

Model successfully saved as 'kerala_weather_model.pkl'!
